In [1]:
import pandas as pd
import requests
import gzip
import io

# 1. Setup your parameters
API_KEY = "pk.cdccbd57622d2ee6db73bb47d102302e"  # Replace with your actual token
NIGERIA_MCC = 621

# OpenCellID full database dump download link format
download_url = f"https://opencellid.org/ocid/downloads?token={API_KEY}&file=cell_towers.csv.gz"

print("Downloading the global cell tower database (this might take a few minutes)...")
response = requests.get(download_url, stream=True)

if response.status_code == 200:
    # 2. Decompress the gzipped content streaming from the API
    with gzip.open(io.BytesIO(response.content), 'rt') as f:
        print("Reading and filtering for Nigerian cell towers (MCC 621)...")

        # Define the OpenCellID standard schema layout
        columns = ['radio', 'mcc', 'net', 'area', 'cell', 'unit', 'lon', 'lat',
                   'range', 'samples', 'changeable', 'created', 'updated', 'averageSignal']

        # Process in chunks to save system RAM
        nigeria_towers = []
        for chunk in pd.read_csv(f, names=columns, header=None, chunksize=100000, low_memory=False):
            # Filter rows where Mobile Country Code matches Nigeria
            ng_chunk = chunk[chunk['mcc'] == NIGERIA_MCC]
            nigeria_towers.append(ng_chunk)

        # Combine all filtered chunks
        nigeria_df = pd.concat(nigeria_towers, ignore_index=True)

        # 3. Save your targeted research asset
        output_file = "nigeria_cell_towers.csv"
        nigeria_df.to_csv(output_file, index=False)
        print(f"Success! Saved {len(nigeria_df)} Nigerian cell towers to '{output_file}'.")
else:
    print(f"Failed to download. Status Code: {response.status_code}. Check your API Key.")

Reading and filtering for Nigerian cell towers (MCC 621)...


/tmp/ipykernel_1397/3154285657.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  nigeria_df = pd.concat(nigeria_towers, ignore_index=True)


Success! Saved 97488 Nigerian cell towers to 'nigeria_cell_towers.csv'.
